#### SCHEMA PROFILE

##### 03.1 DOCUMENTAÇÃO DO SCHEMA PROFILE

###### Objetivo

O **Schema Profile** é responsável por analisar as características estruturais e o preenchimento das colunas de um DataFrame PySpark.

Seu objetivo é produzir um diagnóstico inicial sobre tipos de dados, ausência de valores, diversidade e cardinalidade, independentemente do significado de negócio dos atributos.

O componente utiliza o DataFrame disponibilizado pelo Data Preparation e produz indicadores que podem ser reutilizados por análises posteriores e pelo Data Quality Score.

---

###### Fonte de dados

O componente utiliza `df_prepared`, disponibilizado pelo Notebook 02 — Data Preparation.

As informações estruturais e a quantidade de registros são reutilizadas da etapa anterior para evitar processamento redundante.

---

###### Principais análises

1. **Estrutura:** identificação das colunas e seus tipos Spark.
2. **Preenchimento:** quantidade de valores não NULL.
3. **Valores NULL:** identificação de ausências explícitas.
4. **Valores vazios:** identificação de strings vazias ou compostas apenas por espaços.
5. **Valores NaN:** identificação de valores numéricos não definidos.
6. **Valores distintos:** contagem exata de valores diferentes para tipos elegíveis.
7. **Cardinalidade:** proporção de valores distintos em relação aos valores preenchidos.
8. **Completude efetiva:** percentual de valores que não são NULL, strings vazias ou NaN.
9. **Classificação estrutural:** categorização de preenchimento e cardinalidade.

---

###### Contrato de entrada

| Entrada | Descrição |
|---|---|
| `df_prepared` | DataFrame PySpark validado pelo Data Preparation. |
| `schema_df` | Metadados técnicos das colunas. |
| `total_registros` | Quantidade total de registros. |
| `colunas_numericas` | Lista de colunas numéricas. |
| `colunas_texto` | Lista de colunas textuais. |
| `colunas_data` | Lista de colunas do tipo data. |
| `colunas_data_hora` | Lista de colunas timestamp. |
| `colunas_booleanas` | Lista de colunas booleanas. |
| `colunas_outros` | Lista de colunas de outros tipos. |

---

###### Indicadores calculados

| Indicador | Descrição |
|---|---|
| `preenchidos` | Quantidade de valores não NULL. |
| `null` | Quantidade de valores NULL. |
| `vazios` | Quantidade de strings vazias ou compostas por espaços. |
| `nan` | Quantidade de valores NaN. |
| `distintos` | Quantidade de valores distintos em tipos elegíveis. |
| `pct_preenchimento` | Percentual de valores não NULL. |
| `pct_null` | Percentual de valores NULL. |
| `pct_vazios` | Percentual de strings vazias. |
| `pct_nan` | Percentual de valores NaN. |
| `pct_distintos` | Percentual de valores distintos sobre o total de registros. |
| `pct_cardinalidade` | Percentual de distintos sobre preenchidos. |
| `pct_completude_efetiva` | Percentual de valores efetivamente presentes. |

---

###### Definições dos indicadores

**Percentual de preenchimento**

`pct_preenchimento = preenchidos / total_registros × 100`

Considera preenchidos todos os valores não NULL, inclusive strings vazias e NaN.

**Percentual de NULL**

`pct_null = null / total_registros × 100`

**Percentual de valores vazios**

`pct_vazios = vazios / total_registros × 100`

**Percentual de NaN**

`pct_nan = nan / total_registros × 100`

**Completude efetiva**

`pct_completude_efetiva = (preenchidos - vazios - nan) / total_registros × 100`

**Cardinalidade relativa**

`pct_cardinalidade = distintos / preenchidos × 100`

Quando a análise de distintos não é aplicável ao tipo da coluna, a cardinalidade permanece NULL.

---

###### Classificação de preenchimento

| Condição | Classificação |
|---|---|
| 100% | COMPLETA |
| 95% até menos de 100% | ALTA COMPLETUDE |
| 80% até menos de 95% | COMPLETUDE MODERADA |
| Menos de 80% | BAIXA COMPLETUDE |

A classificação utiliza `pct_preenchimento`, preservando a definição original do framework.

A completude efetiva é apresentada separadamente como indicador complementar.

---

###### Classificação de cardinalidade

| Condição | Classificação |
|---|---|
| 80% ou mais | ALTA CARDINALIDADE |
| 20% até menos de 80% | CARDINALIDADE MODERADA |
| Menos de 20% | BAIXA CARDINALIDADE |
| Métrica indisponível | NAO_APLICAVEL |

Os limites representam faixas descritivas, não critérios universais de qualidade.

Uma coluna com alta cardinalidade não é necessariamente problemática, assim como uma coluna com baixa cardinalidade não é necessariamente inadequada.

---

###### Tratamento de tipos especiais

**Strings:** valores compostos apenas por espaços são contabilizados como vazios.

**Float e Double:** valores NaN são contabilizados separadamente.

**Tipos complexos:** colunas Array, Map e Struct não recebem cardinalidade quando não existe uma política de comparação definida pelo framework.

**Nomes especiais:** referências às colunas utilizam identificadores escapados para evitar interpretação incorreta de pontos e crases.

---

###### Contrato de saída

**DataFrame principal: `schema_profile_df`**

Apresenta uma linha por coluna analisada, contendo métricas estruturais, indicadores percentuais e classificações.

**DataFrames auxiliares:**

- `informacoes_gerais_df`: visão geral da estrutura do DataFrame.
- `visualizacao_preenchimento_null_df`: indicadores de preenchimento e NULL.
- `visualizacao_cardinalidade_df`: indicadores de diversidade e cardinalidade.

---

###### Limitações e interpretação

- A contagem exata de valores distintos pode ter custo elevado em grandes volumes.
- O cálculo consolidado de muitas métricas pode produzir planos Spark extensos em DataFrames muito largos.
- O Schema Profile não identifica automaticamente se um valor é semanticamente correto.
- Valores não NULL não representam necessariamente dados válidos.
- Os limites de classificação não devem ser interpretados como aprovação ou reprovação automática.

---

###### Resultado esperado

Produzir um diagnóstico estrutural completo e reutilizável, com métricas de preenchimento, ausência, diversidade e cardinalidade, preservando o conteúdo original do DataFrame.

Os resultados servem como base para investigação de problemas e para a composição de indicadores de qualidade em etapas posteriores.

> **Princípio arquitetural:** medir as características observáveis dos dados sem presumir que determinado comportamento estrutural representa, por si só, boa ou má qualidade.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 03.3 VALIDAÇÃO DOS CONTRATOS DE ENTRADA
# ============================================================

# O QUE FAZ: 
# Verifica as dependências do Schema Profile.

# COMO FAZ: 
# Valida objetos, contagem e estrutura recebidos.

# POR QUE É IMPORTANTE: 
# Evita falhas silenciosas nos Profiles.

# PERGUNTA RESPONDIDA: 
# "Os insumos estão prontos para análise?"

for nome in ("df_prepared", "schema_df", "total_registros"):
    if nome not in globals():
        raise ValueError(f"Contrato ausente do Data Preparation: {nome}")

if not isinstance(df_prepared, DataFrame):
    raise TypeError("df_prepared deve ser um DataFrame PySpark.")
if not df_prepared.columns:
    raise ValueError("df_prepared não possui colunas.")
if not isinstance(total_registros, int) or total_registros <= 0:
    raise ValueError("total_registros deve ser um inteiro positivo.")
if len(df_prepared.columns) != len(set(df_prepared.columns)):
    raise ValueError("O DataFrame possui nomes de colunas duplicados.")

SCHEMA_DISPLAY_RESULTS = bool(globals().get("DISPLAY_RESULTS", False))
SCHEMA_RUN_TESTS = False
print(f"Contrato validado: {total_registros:,} registros e {len(df_prepared.columns)} colunas.")

In [0]:
# ============================================================
# 03.4 INFORMAÇÕES GERAIS
# ============================================================

# O QUE FAZ: 
# Resume o tamanho e os tipos de coluna.

# COMO FAZ: 
# Reutiliza variáveis do Notebook 02.

# POR QUE É IMPORTANTE: 
# Evita ações Spark redundantes.

# PERGUNTA RESPONDIDA: 
# "Qual a estrutura geral do dataset?"

informacoes_gerais = [
    ("Total de registros", total_registros),
    ("Total de colunas", len(df_prepared.columns)),
    ("Colunas numéricas", len(colunas_numericas)),
    ("Colunas de texto", len(colunas_texto)),
    ("Colunas de data", len(colunas_data)),
    ("Colunas de data/hora", len(colunas_data_hora)),
    ("Colunas booleanas", len(colunas_booleanas)),
    ("Outros tipos", len(colunas_outros)),
]
informacoes_gerais_df = spark.createDataFrame(
    informacoes_gerais,
    T.StructType([
        T.StructField("metrica", T.StringType(), False),
        T.StructField("valor", T.LongType(), False),
    ])
)
print("Informações gerais preparadas.")
if SCHEMA_DISPLAY_RESULTS:
    display(informacoes_gerais_df)

In [0]:
# ============================================================
# 03.5 CONSTRUÇÃO DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Constrói as expressões Spark utilizadas para calcular as métricas estruturais de cada coluna.

# COMO FAZ:
# Cria dinamicamente expressões de agregação para preenchimento, NULL, valores vazios e valores distintos.
# As expressões são construídas antes da execução para permitir que o Spark realize as métricas de forma consolidada.

# POR QUE É IMPORTANTE:
# Evita executar uma ação Spark separada para cada coluna, reduzindo a quantidade de leituras e operações no cluster.

# PERGUNTA RESPONDIDA:
# "Quais métricas estruturais serão calculadas para cada coluna?"

metricas_schema = []
for posicao, campo in enumerate(df_prepared.schema.fields):
    c = dq_col(campo.name)
    tipo = campo.dataType
    prefixo = f"m_{posicao}"

    metricas_schema.append(F.count(c).alias(f"{prefixo}_preenchidos"))
    metricas_schema.append(F.sum(F.when(c.isNull(), 1).otherwise(0)).alias(f"{prefixo}_null"))

    vazio = (c.isNotNull() & (F.trim(c) == "")) if isinstance(tipo, StringType) else F.lit(False)
    metricas_schema.append(F.sum(F.when(vazio, 1).otherwise(0)).alias(f"{prefixo}_vazios"))

    nan = F.isnan(c) if isinstance(tipo, (FloatType, DoubleType)) else F.lit(False)
    metricas_schema.append(F.sum(F.when(nan, 1).otherwise(0)).alias(f"{prefixo}_nan"))

    # dq_is_atomic está disponível no Notebook 00; não exige alteração nele.
    if dq_is_atomic(tipo):
        metricas_schema.append(F.countDistinct(c).alias(f"{prefixo}_distintos"))
    else:
        metricas_schema.append(F.lit(None).cast("long").alias(f"{prefixo}_distintos"))

print(f"{len(metricas_schema)} expressões criadas para {len(df_prepared.columns)} colunas.")

In [0]:
# ============================================================
# 03.6 EXECUÇÃO DO PROFILE
# ============================================================

# O QUE FAZ:
# Executa as métricas construídas anteriormente sobre o DataFrame preparado.

# COMO FAZ:
# Utiliza uma única operação de agregação para calcular, simultaneamente, as métricas estruturais de todas as colunas.

# POR QUE É IMPORTANTE:
# Reduz a quantidade de ações Spark e evita realizar uma leitura independente do DataFrame para cada coluna analisada.

# PERGUNTA RESPONDIDA:
# "Quais são os indicadores estruturais observados no DataFrame?"

schema_metrics_row = df_prepared.agg(*metricas_schema).first()
if schema_metrics_row is None:
    raise RuntimeError("A agregação do Schema Profile não retornou resultado.")
print("Métricas do Schema Profile calculadas com sucesso.")

In [0]:
# ============================================================
# 03.7 CONSTRUÇÃO DO RESULTADO
# ============================================================

# O QUE FAZ:
# Organiza as métricas agregadas em uma estrutura tabular, apresentando uma linha para cada coluna analisada.

# COMO FAZ:
# Recupera os valores das métricas calculadas na agregação anterior e transforma os resultados em registros estruturados.

# POR QUE É IMPORTANTE:
# Padroniza o resultado do Schema Profile e cria uma estrutura que poderá ser utilizada pelos demais componentes do framework, inclusive pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Quais são as características estruturais de cada coluna?"

schema_resultados = []
for posicao, campo in enumerate(df_prepared.schema.fields):
    prefixo = f"m_{posicao}"
    preenchidos = int(schema_metrics_row[f"{prefixo}_preenchidos"] or 0)
    nulos = int(schema_metrics_row[f"{prefixo}_null"] or 0)
    vazios = int(schema_metrics_row[f"{prefixo}_vazios"] or 0)
    nan = int(schema_metrics_row[f"{prefixo}_nan"] or 0)
    distintos_raw = schema_metrics_row[f"{prefixo}_distintos"]
    distintos = int(distintos_raw) if distintos_raw is not None else None

    schema_resultados.append((
        posicao, campo.name, campo.dataType.simpleString(),
        preenchidos, nulos, vazios, distintos,
        100.0 * preenchidos / total_registros,
        100.0 * nulos / total_registros,
        100.0 * vazios / total_registros,
        (100.0 * distintos / total_registros) if distintos is not None else None,
        nan,
        100.0 * nan / total_registros,
        100.0 * (preenchidos - vazios - nan) / total_registros,
    ))
print(f"Resultados preparados para {len(schema_resultados)} colunas.")

In [0]:
# ============================================================
# 03.8 DATAFRAME CONSOLIDADO
# ============================================================

# O QUE FAZ: 
# Publica as métricas em um DataFrame PySpark.

# COMO FAZ: 
# Aplica schema explícito e calcula cardinalidade.

# POR QUE É IMPORTANTE: 
# Entrega resultado estável aos Profiles.

# PERGUNTA RESPONDIDA: 
# "Qual é o perfil estrutural completo?"

schema_result_schema = T.StructType([
    T.StructField("posicao", T.IntegerType(), False),
    T.StructField("coluna", T.StringType(), False),
    T.StructField("tipo_spark", T.StringType(), False),
    T.StructField("preenchidos", T.LongType(), False),
    T.StructField("null", T.LongType(), False),
    T.StructField("vazios", T.LongType(), False),
    T.StructField("distintos", T.LongType(), True),
    T.StructField("pct_preenchimento", T.DoubleType(), False),
    T.StructField("pct_null", T.DoubleType(), False),
    T.StructField("pct_vazios", T.DoubleType(), False),
    T.StructField("pct_distintos", T.DoubleType(), True),
    T.StructField("nan", T.LongType(), False),
    T.StructField("pct_nan", T.DoubleType(), False),
    T.StructField("pct_completude_efetiva", T.DoubleType(), False),
])

schema_profile_df = (
    spark.createDataFrame(schema_resultados, schema_result_schema)
    .withColumn(
        "pct_cardinalidade",
        F.when(
            (F.col("preenchidos") > 0) & F.col("distintos").isNotNull(),
            100.0 * F.col("distintos") / F.col("preenchidos")
        ).otherwise(F.lit(None).cast("double"))
    )
    .orderBy("posicao")
)
print("schema_profile_df criado com sucesso.")


In [0]:
# ============================================================
# 03.9 EXIBIÇÃO DO SCHEMA PROFILE
# ============================================================

# O QUE FAZ:
# Exibe o resultado consolidado do Schema Profile.

# COMO FAZ:
# Utiliza a visualização nativa do Databricks sobre o DataFrame Spark produzido na etapa anterior.

# POR QUE É IMPORTANTE:
# Permite validar visualmente as métricas estruturais antes das etapas de visualização e classificação.

# PERGUNTA RESPONDIDA:
# "Como estão distribuídas as principais características estruturais das colunas?"

if SCHEMA_DISPLAY_RESULTS:
    display(schema_profile_df)
else:
    print("Exibição desabilitada: schema_profile_df disponível para uso.")

In [0]:
# ============================================================
# 03.10 VISUALIZAÇÃO — PREENCHIMENTO X NULL
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de preenchimento e NULL para visualização comparativa por coluna.

# COMO FAZ:
# Seleciona somente as métricas percentuais necessárias e reorganiza os dados para facilitar a construção de um gráfico de barras no Databricks.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente colunas com baixo preenchimento ou elevada ocorrência de valores NULL.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam problemas de preenchimento?"

visualizacao_preenchimento_null_df = schema_profile_df.select(
    "coluna", "pct_preenchimento", "pct_null"
)
if SCHEMA_DISPLAY_RESULTS:
    display(visualizacao_preenchimento_null_df)
else:
    print("Visualização de preenchimento/NULL preparada (exibição desabilitada).")

In [0]:
# ============================================================
# 03.11 VISUALIZAÇÃO — CARDINALIDADE
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de cardinalidade das colunas para visualização.

# COMO FAZ:
# Seleciona a coluna e o percentual de cardinalidade calculado pelo Schema Profile.

# POR QUE É IMPORTANTE:
# A visualização permite identificar colunas com alta ou baixa diversidade relativa de valores.

# PERGUNTA RESPONDIDA:
# "Quais colunas possuem maior ou menor diversidade de valores?"

visualizacao_cardinalidade_df = (
    schema_profile_df.select("coluna", "distintos", "pct_cardinalidade")
    .orderBy(F.col("pct_cardinalidade").desc_nulls_last())
)
if SCHEMA_DISPLAY_RESULTS:
    display(visualizacao_cardinalidade_df)
else:
    print("Visualização de cardinalidade preparada (exibição desabilitada).")

In [0]:
# ============================================================
# 03.12 CLASSIFICAÇÃO ESTRUTURAL DAS COLUNAS
# ============================================================

# O QUE FAZ:
# Classifica as colunas de acordo com suas características estruturais observadas no Schema Profile.

# COMO FAZ:
# Utiliza indicadores de preenchimento e cardinalidade para identificar características como colunas completas, incompletas, de baixa cardinalidade e de alta cardinalidade.

# POR QUE É IMPORTANTE:
# Transforma métricas estatísticas em informações interpretáveis que poderão ser utilizadas posteriormente pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é a característica estrutural predominante de cada coluna?"

schema_profile_df = (
    schema_profile_df
    .withColumn(
        "classificacao_preenchimento",
        F.when(F.col("pct_preenchimento") == 100, "COMPLETA")
         .when(F.col("pct_preenchimento") >= 95, "ALTA COMPLETUDE")
         .when(F.col("pct_preenchimento") >= 80, "COMPLETUDE MODERADA")
         .otherwise("BAIXA COMPLETUDE")
    )
    .withColumn(
        "classificacao_cardinalidade",
        F.when(F.col("pct_cardinalidade").isNull(), "NAO_APLICAVEL")
         .when(F.col("pct_cardinalidade") >= 80, "ALTA CARDINALIDADE")
         .when(F.col("pct_cardinalidade") >= 20, "CARDINALIDADE MODERADA")
         .otherwise("BAIXA CARDINALIDADE")
    )
)
print("Classificações estruturais calculadas.")
if SCHEMA_DISPLAY_RESULTS:
    display(schema_profile_df)

In [0]:
# ============================================================
# 03.13 TESTES OPCIONAIS DO CONTRATO DE SAÍDA
# ============================================================

# O QUE FAZ: 
# Verifica o resultado final do Profile.

# COMO FAZ: 
# Executa verificações com expressões Spark.

# POR QUE É IMPORTANTE: 
# Detecta erros de estrutura e percentuais.

# PERGUNTA RESPONDIDA: 
# "O resultado atende ao contrato esperado?"

if SCHEMA_RUN_TESTS:
    assert isinstance(schema_profile_df, DataFrame)
    assert schema_profile_df.count() == len(df_prepared.columns)
    assert schema_profile_df.filter(
        (F.col("pct_preenchimento") < 0) |
        (F.col("pct_preenchimento") > 100) |
        (F.col("pct_completude_efetiva") < 0) |
        (F.col("pct_completude_efetiva") > 100)
    ).limit(1).count() == 0
    assert schema_profile_df.filter(
        F.abs(F.col("pct_preenchimento") + F.col("pct_null") - 100) > 1e-8
    ).limit(1).count() == 0
    print("[OK] Contrato de saída, número de colunas e percentuais validados.")
else:
    print("Testes desabilitados. Defina SCHEMA_RUN_TESTS = True na célula 03.3.")
